In [4]:
# required imports for data cleaning
import os
import numpy as np
from tqdm import tqdm
from pathlib import Path
import pandas as pd
import subprocess  
from pathlib import Path 
import nibabel as nib
import matplotlib.pyplot as plt
from typing import Tuple
import scipy.ndimage as ndi
import SimpleITK as sitk

In [5]:
#skull stripping functions

FREESURFER_HOME = "/usr/local/freesurfer/8.1.0"

def run_synthstrip(in_file, out_file):
    """
    Run SynthStrip on a NIFTI file. 
    Takes in a file path and saves the stripped skull as a new file to the specified path
    """
    command = f"""
    export FREESURFER_HOME={FREESURFER_HOME}
    source $FREESURFER_HOME/SetUpFreeSurfer.sh
    $FREESURFER_HOME/bin/mri_synthstrip -i "{in_file}" -o "{out_file}"
    """
    subprocess.run(command, shell=True, executable="/bin/bash", check=True)

def skullstrip_paths(paths, output_dir):
    """
    Run SynthStrip on a list of MRI paths and return failed paths.
    """
    output_base = Path(output_dir)
    output_base.mkdir(parents=True, exist_ok=True)

    failed_paths = []

    for input_path in tqdm(paths, desc="Skull stripping MRIs", unit="scan"):
        input_file = Path(input_path)
        output_file = output_base / f"{input_file.stem}_brain.nii.gz"

        if output_file.exists():
            continue

        try:
            run_synthstrip(input_file, output_file)
        except Exception as e:
            failed_paths.append(str(input_file))
            
    return failed_paths

In [6]:
TARGET_SPACING = (1.0, 1.0, 1.0)
TARGET_SHAPE = (160, 192, 160)
CLIP_PERCENTILES = (0.5, 99.5)
EPS = 1e-8

# Utility functions post skull stripping
def load_nifti(path: str) -> Tuple[np.ndarray, Tuple[float, float, float]]:
    """
    Load skull stripped NIfTI file and return data + voxel spacing
    Ensure orientation is RAS (Right-Anterior-Superior)
    Returns:
        volume: np.ndarray (D, H, W)
        spacing: tuple (sx, sy, sz)
    """
    nii = nib.load(path)
    nii = nib.as_closest_canonical(nii) 
    volume = nii.get_fdata().astype(np.float32)
    spacing = nii.header.get_zooms()[:3]
    return volume, spacing


def n4_bias_correction(volume: np.ndarray) -> np.ndarray:
    """
    Apply N4 bias field correction using SimpleITK.
    Corrects smooth intensity inhomogeneities caused by scanner field imperfections
    """
    sitk_image = sitk.GetImageFromArray(volume)
    sitk_image = sitk.Cast(sitk_image, sitk.sitkFloat32)
    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    corrected = corrector.Execute(sitk_image)
    return sitk.GetArrayFromImage(corrected)


def resample_volume(volume: np.ndarray, spacing: Tuple[float, float, float], target_spacing: Tuple[float, float, float]) -> np.ndarray:
    """
    Resample volume to target voxel spacing using interpolation
    This ensures all subjects have the same physical resolution
    """
    z = np.array(spacing) / np.array(target_spacing)
    return ndi.zoom(volume, zoom=z, order=3)


def robust_intensity_normalization(volume: np.ndarray) -> np.ndarray:
    """
    Robust z-score normalization
    1. Clip extreme intensities
    2. Normalize to zero mean and unit variance
    """
    lo, hi = np.percentile(volume, CLIP_PERCENTILES)
    volume = np.clip(volume, lo, hi)
    mean = volume.mean()
    std = volume.std() + EPS
    return (volume - mean) / std


def center_crop_or_pad(volume: np.ndarray, target_shape: Tuple[int, int, int]) -> np.ndarray:
    """
    Center-crop or zero-pad the volume to the target shape.
    """
    out = np.zeros(target_shape, dtype=np.float32)
    src_shape = np.array(volume.shape)
    tgt_shape = np.array(target_shape)

    start_src = np.maximum((src_shape - tgt_shape) // 2, 0)
    end_src = start_src + np.minimum(src_shape, tgt_shape)

    start_tgt = np.maximum((tgt_shape - src_shape) // 2, 0)
    end_tgt = start_tgt + np.minimum(src_shape, tgt_shape)

    out[
        start_tgt[0]:end_tgt[0],
        start_tgt[1]:end_tgt[1],
        start_tgt[2]:end_tgt[2],
    ] = volume[
        start_src[0]:end_src[0],
        start_src[1]:end_src[1],
        start_src[2]:end_src[2],
    ]

    return out

def preprocess_nifti(nifti_path: str) -> np.ndarray:
    """
    Full preprocessing pipeline for a single MRI scan.
    Args:
        nifti_path: path to .nii or .nii.gz
    Returns:
        volume: np.ndarray (1, D, H, W) ready for PyTorch
    """
    volume, spacing = load_nifti(nifti_path)

    volume = n4_bias_correction(volume)

    volume = resample_volume(volume, spacing, TARGET_SPACING)

    volume = robust_intensity_normalization(volume)

    volume = center_crop_or_pad(volume, TARGET_SHAPE)

    volume = volume[None, ...]  # (1, D, H, W)

    return volume.astype(np.float32)

In [7]:
def preprocess_mri_paths(paths, output_dir):
    """
    Preprocess the skull-stripped NIfTI MRIs and save them as .npy files

    Parameters
    ----------
    paths : list
        List of NIfTI file paths
    output_dir : str
        Directory where preprocessed .npy files will be saved

    Returns
    -------
    failed_paths : list
        List of paths that failed preprocessing
    """

    os.makedirs(output_dir, exist_ok=True)
    failed_paths = []

    for path in tqdm(paths, desc="Preprocessing MRIs", unit="scan"):
        try:
            base = os.path.basename(path).replace(".nii.gz", "")
            save_path = os.path.join(output_dir, base + ".npy")

            if os.path.exists(save_path):
                continue

            volume = preprocess_nifti(path)  

            np.save(save_path, volume)

        except Exception as e:
            failed_paths.append(path)
            continue
            
    return failed_paths

In [8]:
#visualize MRI scan to visually check results

def show_mri(path):
    img = nib.load(path)
    data = img.get_fdata()

    x = data.shape[0] // 2
    y = data.shape[1] // 2
    z = data.shape[2] // 2

    fig, axes = plt.subplots(1,3, figsize=(12,4))

    axes[0].imshow(np.rot90(data[x,:,:]), cmap="gray")
    axes[0].set_title("Sagittal")

    axes[1].imshow(np.rot90(data[:,y,:]), cmap="gray")
    axes[1].set_title("Coronal")

    axes[2].imshow(np.rot90(data[:,:,z]), cmap="gray")
    axes[2].set_title("Axial")

    for ax in axes:
        ax.axis("off")

    plt.show()

In [ ]:
# Remove erroneous 

def safe_load(f):
    x = np.load(f)
    return np.squeeze(x)


def center_of_mass_shift(x):
    if len(x.shape) == 3:
        img = np.mean(x, axis=0)
    elif len(x.shape) == 2:
        img = x
    else:
        return None

    img = img - np.min(img)
    if np.sum(img) == 0:
        return 0

    img = img / np.sum(img)

    coords = np.indices(img.shape)
    cy = np.sum(coords[0] * img)
    cx = np.sum(coords[1] * img)

    return np.sqrt((cy - img.shape[0]/2)**2 + (cx - img.shape[1]/2)**2)


def edge_ratio(x):
    if len(x.shape) == 3:
        mid = x.shape[0] // 2
        img = x[mid]
    else:
        img = x

    border = np.concatenate([
        img[0, :], img[-1, :], img[:, 0], img[:, -1]
    ])

    return np.mean(border == 0)


def find_bad_files(files, shift_percentile=95, edge_thr=0.2, z_thresh=2):
    stats = []
    means, stds = [], []

    bad_all_load = []
    bad_stat = []
    bad_shift = []
    bad_edge = []

    for f in files:
        try:
            x = np.load(f)

            s = {
                "file": f,
                "shape": x.shape,
                "nan": np.isnan(x).any(),
                "inf": np.isinf(x).any(),
                "mean": float(np.mean(x)),
                "std": float(np.std(x))
            }

            stats.append(s)
            means.append(s["mean"])
            stds.append(s["std"])

        except:
            bad_all_load.append(f)

    means = np.array(means)
    stds = np.array(stds)

    def z(x):
        return (x - x.mean()) / (x.std() + 1e-8)

    mean_z = z(means)
    std_z = z(stds)

    stat_idx = np.where((np.abs(mean_z) > z_thresh) | (np.abs(std_z) > z_thresh))[0]
    bad_stat = [stats[i]["file"] for i in stat_idx]

    shift_scores = []
    edge_scores = []

    for s in stats:
        x = safe_load(s["file"])
        shift_scores.append(center_of_mass_shift(x))
        edge_scores.append(edge_ratio(x))

    shift_scores = np.array(shift_scores)
    edge_scores = np.array(edge_scores)

    shift_thr = np.percentile(shift_scores, shift_percentile)

    bad_shift = [stats[i]["file"] for i in np.where(shift_scores > shift_thr)[0]]
    bad_edge = [stats[i]["file"] for i in np.where(edge_scores > edge_thr)[0]]

    bad_all = set(bad_all_load) | set(bad_stat) | set(bad_shift) | set(bad_edge)

    return {
        "bad_all": bad_all,
        "bad_load": bad_all_load,
        "bad_stat": bad_stat,
        "bad_shift": bad_shift,
        "bad_edge": bad_edge,
        "stats": stats
    }



def permanently_filter_dataset(dataset_path, bad_files, path_col="fixed_path"):
    
    df = pd.read_csv(dataset_path) 

    bad_files = set(bad_files)

    original_len = len(df)

    df_clean = df[~df[path_col].isin(bad_files)].reset_index(drop=True)

    df_clean.to_csv(dataset_path, index=False)  

    print(f"Original rows: {original_len}")
    print(f"Remaining rows: {len(df_clean)}")
    print(f"Removed rows: {original_len - len(df_clean)}")

    return df_clean
